# M7 Run 3 - A/B imbalance (focal vs balanced sampling)

Disentangles the baseline's **double correction** (balanced sampling + focal). **2x2 factorial**: sampler {balanced, natural} x loss {focal, BCE} -> `both` (baseline) / `samp` / `focal` / `wbce`.

**Fast mini-OOF proxy** (ranking only): train on folds {5,6,7,8}, test pooled on folds {1,2,3,4} (~58 WPW), 3 seeds, @5000. **Reuses the Run 2 cache** (no re-extraction). Pre-registered bar: a variant wins only if its AP CI lower bound exceeds the baseline AP; else keep `both`. Winner is confirmed later in full OOF.

> Mini-OOF trains on 4 folds only -> absolute AP < full-OOF 0.644; the **ranking** is the signal, not the level.

### Block 3.0: Setup and A/B grid

In [ ]:
# M7 Run 3 - A/B IMBALANCE: disentangle focal vs balanced sampling. 2x2 factorial:
#   sampler {balanced, natural} x loss {focal, weighted-BCE/BCE}  ->  both / samp / focal / wbce.
# Fast mini-OOF proxy (ranking only): train on folds {5,6,7,8}, test pooled on folds {1,2,3,4} (~58 WPW), 3 seeds.
# Resolution 5000. Reuses the Run 2 cache (no re-extraction). Winner confirmed later in full OOF.
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings('ignore')
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
METRICS=os.path.join(ROOT,'reports','metrics'); MODELS=os.path.join(ROOT,'models','M7_run3')
CACHE_DIR=os.path.join(PROCESSED,'m7_run2_cache')      # reuse Run 2 cache
for d in (METRICS,MODELS): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC)
from evaluation import _boot_ci
RANDOM_STATE=42; RESO=5000
SEEDS=[0,1,2]                              # mini-OOF: 3 seeds for ranking
EPOCHS=60; PATIENCE=10; BATCH=64; LR=1e-3; WD=1e-4
FOCAL_GAMMA=2.0; FOCAL_ALPHA=0.5; VAL_FRAC=0.15; N_NEG_TRAIN=12000; N_JOBS=6
TEST_FOLDS=[1,2,3,4]; TRAIN_FOLDS=[5,6,7,8]
# (name, sampler, loss): 2x2 factorial. 'both' = Run 1/2 baseline (balanced + focal).
VARIANTS=[('both','balanced','focal'),('samp','balanced','bce'),
          ('focal','natural','focal'),('wbce','natural','bce')]
REF_OOF_BASELINE=0.644                     # Run 2 full-OOF 'both' (context; mini-OOF is on less train data)
print('M7 Run 3 | A/B imbalance 2x2 | mini-OOF train %s -> test %s | seeds %s @ %d'%(TRAIN_FOLDS,TEST_FOLDS,SEEDS,RESO))
print('variants:', [v[0] for v in VARIANTS])
print('Block 3.0 OK.')


### Block 3.1: Reuse Run 2 cache + mini-OOF split

In [ ]:
# Reuse Run 2 cache (folds 1-8, native 5000, z-scored, float16). Build mini-OOF train/test masks.
z=np.load(os.path.join(CACHE_DIR,'m7_run2_data.npz'),allow_pickle=True)
X18=z['X18']; fold18=z['fold18']; y18=z['y18']
tr_mask=np.isin(fold18,TRAIN_FOLDS); te_mask=np.isin(fold18,TEST_FOLDS)
tr_all=np.where(tr_mask)[0]; te_idx=np.where(te_mask)[0]
yte=y18[te_idx]; Xte=X18[te_idx]
print('mini-OOF | train pool %d (%d WPW) | test %d (%d WPW, true prevalence)'%(
    tr_mask.sum(),int(y18[tr_mask].sum()),te_mask.sum(),int(yte.sum())))
print('Block 3.1 OK.')


### Block 3.2: 1D-ResNet (identical to Run 1/2)

In [ ]:
# 1D-ResNet (modest) - identical to Run 1/2 (frozen architecture).
import torch, torch.nn as nn
torch.set_num_threads(N_JOBS)
class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)
class ResNet1d(nn.Module):
    def __init__(self,ch=(16,32,64),pdrop=0.3):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1); self.layer2=BasicBlock1d(ch[0],ch[1],2); self.layer3=BasicBlock1d(ch[1],ch[2],2)
        self.pool=nn.AdaptiveMaxPool1d(1); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(pdrop),nn.Linear(ch[2],1))
    def forward(self,x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x)))))).squeeze(1)
print('Block 3.2 OK | params:', sum(p.numel() for p in ResNet1d().parameters()))


### Block 3.3: Loss/sampler modes + training

In [ ]:
# Loss + augmentation + training with SAMPLER and LOSS modes (the A/B axes). Early-stop on val (same loss as train).
def focal_loss(logits,targets,gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA):
    ce=nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=torch.where(targets==1,p,1-p)
    a=torch.where(targets==1,torch.full_like(targets,alpha),torch.full_like(targets,1-alpha))
    return (a*(1-pt).pow(gamma)*ce).mean()
def make_loss(kind,pos_weight):
    if kind=='focal': return lambda lo,t: focal_loss(lo,t)
    pw=None if pos_weight is None else torch.tensor([float(pos_weight)])
    bce=nn.BCEWithLogitsLoss(pos_weight=pw)
    return lambda lo,t: bce(lo,t)
def augment(xb):
    T=xb.shape[2]; mx=max(1,int(0.04*T))
    xb=torch.roll(xb,int(torch.randint(-mx,mx+1,(1,)).item()),dims=2)
    xb=xb*(0.8+0.4*torch.rand(xb.shape[0],1,1)); xb=xb+0.02*torch.randn_like(xb)
    t=torch.linspace(0,1,T).view(1,1,-1)
    fb=0.5+2.0*torch.rand(xb.shape[0],1,1); ph=2*np.pi*torch.rand(xb.shape[0],1,1)
    xb=xb+0.05*torch.sin(2*np.pi*fb*t+ph)
    if torch.rand(1).item()<0.3: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    return xb
def predict(model,X):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),256):
            out.append(torch.sigmoid(model(torch.tensor(np.ascontiguousarray(X[i:i+256],dtype=np.float32)))).numpy())
    return np.nan_to_num(np.concatenate(out))
def train_one(seed,Xtr,ytr,Xva,yva,sampler,loss_kind,pos_weight):
    torch.manual_seed(seed); np.random.seed(seed); rng=np.random.default_rng(seed)
    pos=np.where(ytr==1)[0]; neg=np.where(ytr==0)[0]; half=BATCH//2; N=len(ytr)
    lossf=make_loss(loss_kind,pos_weight)
    Xv=torch.tensor(np.ascontiguousarray(Xva,dtype=np.float32)); yv=torch.tensor(yva)
    model=ResNet1d(); opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
    steps=max(1,N//BATCH); best=1e9; best_state=None; bad=0
    for ep in range(EPOCHS):
        model.train()
        for _ in range(steps):
            if sampler=='balanced':
                bi=np.concatenate([rng.choice(pos,half,True),rng.choice(neg,half,True)])
            else:
                bi=rng.integers(0,N,BATCH)                       # natural (true imbalance)
            xb=augment(torch.tensor(np.ascontiguousarray(Xtr[bi],dtype=np.float32))); yb=torch.tensor(ytr[bi])
            opt.zero_grad(); loss=lossf(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=[lossf(model(Xv[i:i+256]),yv[i:i+256]).item() for i in range(0,len(Xv),256)]
        vloss=float(np.mean(vl))
        if vloss<best-1e-4: best=vloss; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state); return model
print('Block 3.3 OK.')


### Block 3.4: A/B loop (mini-OOF, checkpointed per variant+seed)

In [ ]:
# A/B loop (mini-OOF). Per variant: build train pool (all train WPW + subsampled negs), seeded stratified internal
# val, 3-seed ensemble, predict test folds 1-4 (true prevalence). Checkpoint per (variant, seed).
def build_pool(seed0):
    rng=np.random.default_rng(seed0)
    pos=tr_all[y18[tr_all]==1]; negall=tr_all[y18[tr_all]==0]
    neg=rng.choice(negall,min(N_NEG_TRAIN,len(negall)),replace=False)
    vp=pos.copy(); vn=neg.copy(); rng.shuffle(vp); rng.shuffle(vn)
    nvp=max(6,int(VAL_FRAC*len(vp))); nvn=int(VAL_FRAC*len(vn))
    val=np.concatenate([vp[:nvp],vn[:nvn]]); pool=np.concatenate([pos,neg]); tr=np.setdiff1d(pool,val)
    return tr,val,float(len(neg))/max(len(pos)-nvp,1)
tr_idx,val_idx,pw_ratio=build_pool(2024)            # fixed split shared across variants (fair A/B)
Xtr=np.ascontiguousarray(X18[tr_idx],dtype=np.float32); ytr=y18[tr_idx]
Xva=X18[val_idx]; yva=y18[val_idx]
print('pool: train %d (%d WPW) | val %d (%d WPW) | pos_weight(wbce)=%.0f'%(
    len(tr_idx),int(ytr.sum()),len(val_idx),int(yva.sum()),pw_ratio))
CK=os.path.join(MODELS,'run3_ab_ckpt.npz')
store={} 
if os.path.exists(CK):
    d=np.load(CK,allow_pickle=True); store=d['store'].item(); print('[ckpt] resumed variants:',list(store.keys()))
rows=[]; t0=time.time()
for name,sampler,loss_kind in VARIANTS:
    pw=pw_ratio if (sampler=='natural' and loss_kind=='bce') else None
    key=name
    seed_pred=store.get(key,{}).get('pred',[]); done=store.get(key,{}).get('done',0)
    seed_pred=[np.asarray(p) for p in seed_pred]
    for si,s in enumerate(SEEDS):
        if si<done: continue
        model=train_one(s,Xtr,ytr,Xva,yva,sampler,loss_kind,pw)
        seed_pred.append(predict(model,Xte))
        store[key]={'pred':[p.tolist() for p in seed_pred],'done':si+1}
        np.savez(CK,store=np.array(store,dtype=object))
    ens=np.mean(seed_pred,0); ap=float(average_precision_score(yte,ens)); auc=float(roc_auc_score(yte,ens))
    lo,hi=_boot_ci(yte.astype(int),ens,average_precision_score)
    seed_ap=[float(average_precision_score(yte,p)) for p in seed_pred]
    rows.append(dict(variant=name,sampler=sampler,loss=loss_kind,AP=ap,AP_lo=lo,AP_hi=hi,AUC=auc,
                     AP_seed_mean=float(np.mean(seed_ap)),AP_seed_std=float(np.std(seed_ap))))
    print('  [%-5s] %s+%-5s AP %.3f CI[%.3f,%.3f] | per-seed %.3f+/-%.3f | %.1f min'%(
        name,sampler,loss_kind,ap,lo,hi,np.mean(seed_ap),np.std(seed_ap),(time.time()-t0)/60))
ab=pd.DataFrame(rows)
print('Block 3.4 OK.'); print(ab.to_string(index=False))


### Block 3.5: Ranking + decision (pre-registered CI bar)

In [ ]:
# Ranking + decision. Pre-registered bar: a variant beats the baseline only if its AP CI lower bound exceeds
# the baseline AP (beyond bootstrap noise); else keep the simplest / current baseline ('both').
base=ab[ab.variant=='both'].iloc[0]
ab_sorted=ab.sort_values('AP',ascending=False).reset_index(drop=True)
winner=ab_sorted.iloc[0]
beats=bool(winner.AP_lo>base.AP and winner.variant!='both')
chosen=winner.variant if beats else 'both'
print('=== M7 Run 3 - imbalance A/B (mini-OOF, test folds 1-4) ===')
print(ab_sorted.to_string(index=False))
print('  baseline both: AP %.3f CI[%.3f,%.3f]'%(base.AP,base.AP_lo,base.AP_hi))
print('  top: %s AP %.3f CI[%.3f,%.3f]'%(winner.variant,winner.AP,winner.AP_lo,winner.AP_hi))
print('  beats baseline beyond CI ? %s -> ADOPT: %s'%(beats,chosen))
print('  NOTE: mini-OOF trains on 4 folds only (ranking proxy). Absolute AP < full-OOF 0.644; ranking is the signal.')
json.dump({'variants':ab_sorted.to_dict('records'),'baseline_AP':float(base.AP),
           'winner':winner.variant,'beats_beyond_CI':beats,'adopted':chosen},
          open(os.path.join(METRICS,'m7_run3_imbalance_ab.json'),'w'),indent=2,default=float)
print('Block 3.5 OK | adopted imbalance config: %s'%chosen)
